### 训练CNN与CNN架构
#### CNN的组成部分
除了卷积层，池化层和全连接层，CNN的组成部分还包括Normalization Layers，Dropout和激活函数
##### 标准化层（Normalization Layers）
标准化包括Batch Norm，Layer Norm，Instance Norm，Group Norm等，其主要作用是稳定中间层数据分布，让训练更快、更稳，同时在一定程度上缓解过拟合。<br>
设输入为x：(N, C, H, W)，则**Layer Norm**对批次内的每个样本，计算D = C\*H\*W个数据的平均值$\mu$和标准差$\sigma$，然后输出$Y = \frac{\gamma(x-\mu)}{\sigma}+\beta$，其中$\gamma$和$\beta$是可学习的参数，用于调整分布；**Batch Norm**则是对C中的每个通道，对N\*H\*W个数据做上述操作；此外，还有Instance Norm和Group Norm
<div align="center">
  <img src="class_images/normalization.jpg" width="800">
</div>

#### Dropout
作用：这是一种正则化方式，使得模型在训练集上更难学，但是在测试集上有更好的泛化能力<br>
具体实现方式：在前向传播中，随机将一些神经元的值设为$0$，设为$0$的概率是一个超参数，通常为$0.5$<br>
注意，在测试集上作用时，所用神经元都是有用的，这会导致下一层的输入个数多于训练时模型的输入个数，因此需要乘以前面的dropout概率，但现代框架会在训练时把保留下来的神经元放大$\frac{1}{p}$，因此测试时不需要这个步骤

In [ ]:
# dropout implementation from scratch
import torch

p = 0.5

def train_step(x, W1, b1, W2, b2, W3, b3):
    H1 = torch.max(torch.zeros_like(x.mm(W1) + b1), x.mm(W1) + b1)
    U1 = torch.rand_like(H1) < p # dropout mask: 生成一个与H1同形状的取值0到1的随机矩阵，元素为True的概率为p
    H1 *= U1 # dropout: 将H1中对应于True的元素保留，其他元素置为0
    H2 = torch.max(torch.zeros_like(H1.mm(W2) + b2), H1.mm(W2) + b2)
    U2 = torch.rand_like(H2) < p
    H2 *= U2
    y = H2.mm(W3) + b3
    return U1, H1, y

def predict(x, W1, b1, W2, b2, W3, b3):
    H1 = torch.max(torch.zeros_like(x.mm(W1) + b1), x.mm(W1) + b1) * p # 在预测阶段，直接将H1乘以p来近似dropout的效果
    H2 = torch.max(torch.zeros_like(H1.mm(W2) + b2), H1.mm(W2) + b2) * p
    y = H2.mm(W3) + b3
    return y

x= torch.rand(3, 4)
W1 = torch.rand(4, 5)
b1 = torch.rand(5)
W2 = torch.rand(5, 6)
b2 = torch.rand(6)
W3 = torch.rand(6, 1)
b3 = torch.rand(1)

U1, H1, y = train_step(x, W1, b1, W2, b2, W3, b3)
print(x)
print(U1)
print(H1)
print(y)

tensor([[0.6232, 0.2061, 0.0390, 0.6328],
        [0.1560, 0.9724, 0.8827, 0.5390],
        [0.6439, 0.3755, 0.8148, 0.4365]])
tensor([[ True, False, False, False,  True],
        [ True,  True, False,  True,  True],
        [False,  True,  True,  True, False]])
tensor([[1.3042, 0.0000, 0.0000, 0.0000, 1.3700],
        [2.5262, 2.2764, 0.0000, 1.9091, 1.9127],
        [0.0000, 2.0448, 1.8079, 1.6359, 0.0000]])
tensor([[2.2113],
        [7.7111],
        [4.5948]])


In [ ]:
from torch import nn

dropout = nn.Dropout(0.5)

#### 激活函数（Activate Functions）
激活函数的作用是在模型中引入非线性性。sigmoid函数曾经是流行的选择，但它的问题是在经过多层神经网络后，当sigmoid函数的输入很大或很小时，可能出现梯度消失的问题；ReLU是当下更多的选择，问题是当输入小于0时失效，一类新的激活函数是对ReLU的拟合，使其在0点附近光滑（例如GeLU）<br>
CNN中，激活函数出现在线性层之后，包括前向传播中的全连接层，卷积层等
#### CNN架构
早期经典架构包括AlexNet和VGGNet，使用较小卷积核的原因，例如3*3：三层3\*3卷积层的感受野和一个7\*7卷积层的感受野相同，但更深的神经网络意味着更多的非线性性，并且有更少的参数（$3\times(3^2 C^2)<7^2 C^2$）<br>
ResNet的出现：研究发现更浅的CNN比更深的在测试集和训练集上均效果更好，这不是过拟合导致的，而是更深的CNN更难优化。因此，一个解决方案是用神经网络层学习残差映射，而不是整个目标映射，通俗的说，或许目标只是在输入上改一点点，因此ResNet只需要先将输入原封不动的搬过去，然后学习那一点点残差就好，但普通的CNN需要费很大劲来从头拟合这个接近恒等映射的映射
<div align="center">
  <img src="class_images/AlexNet_VGGNet.jpg" width="30%">
  <img src="class_images/residual_connection.jpg" width="60%">
</div>

<div align="center">
  <img src="class_images/ResNet.jpg" width="800">
</div>

#### 权重初始化
过大或过小权重初始化会导致输出过大或过小，从而难以训练，一种好的权重初始化方法是Kaiming/MSRA Initialization

In [ ]:
import math

w = torch.randn(Din, Dout) * math.sqrt(2.0 / Din)
b = torch.zeros(Dout)

w = torch.empty(Din, Dout)
torch.nn.init.kaiming_normal_(w, mode='fan_in', nonlinearity='relu')

b = torch.zeros(Dout)

#### 训练CNN
##### 数据预处理（Data Preprocessing）

In [ ]:
norm_pixel[i, j, c] = (pixel[i, j, c] - np.mean(pixel[:, :, c])) / np.std(pixel[:, :, c])

##### 数据增强（Data Augmentation）
数据增强可以理解为向数据集加入噪音，从而达到正则化目的，防止过拟合。数据增强的具体操作包括：水平翻转（Horizontal Flips）、垂直翻转（Vertical Flips）、随机裁剪、Color Jitter（包括随机对比度和亮度）、添加遮挡
##### 迁移学习（Transfer Learning）
在较小数据集上训练时，将在大数据集（Imagenet）上训练好的模型直接搬过来，保留特征提取部分，只对最后的线性分类器做修改，然后训练；对于更大的数据集，将在大数据集（Imagenet）上训练好的模型参数作为初始化进行微调，通常加入更多神经网络层。迁移学习主要在当前数据集和原数据集较为相似时起作用
##### 超参数的选择（Choosing Hyperparameters）
1. 检查初始loss，确保loss合理
2. 先在一个小数据集上训练，主要为了观察模型是否真的在学习
3. 确定合适的学习率，可以尝试的学习率包括：1e-1，1e-2，1e-3，1e-4，1e-5
4. 粗糙地选择一系列超参数，训练1-5个epochs
5. 进一步确定超参数，训练更久
6. 看损失曲线和准确率曲线
* 在尝试超参数时，随机选择超参数往往比网格式搜索更有效